In [2]:
from sklearn.feature_extraction.text import CountVectorizer

v=CountVectorizer(ngram_range=(1,2))
v.fit(["Thor Hathodawals is looking for a job"])
v.vocabulary_

{'thor': 9,
 'hathodawals': 2,
 'is': 4,
 'looking': 7,
 'for': 0,
 'job': 6,
 'thor hathodawals': 10,
 'hathodawals is': 3,
 'is looking': 5,
 'looking for': 8,
 'for job': 1}

In [3]:
corpus=[
    "Thor ate pizza",
    "Loki is tail",
    "Loki is eating pizza"
]

In [11]:
import spacy
nlp=spacy.load("en_core_web_sm")

def preprocess(text):
  doc=nlp(text)

  filtered_tokens=[]

  for token in doc:
    if token.is_stop or token.is_punct:
      continue
    filtered_tokens.append(token.lemma_)

  return " ".join(filtered_tokens)

preprocess("loki is eating pizza")

'loki eat pizza'

In [12]:
corpus_processed=[preprocess(text) for text in corpus]
corpus_processed

['thor eat pizza', 'Loki tail', 'Loki eat pizza']

In [13]:
v=CountVectorizer(ngram_range=(1,2))
v.fit(corpus_processed)
v.vocabulary_

{'thor': 7,
 'eat': 0,
 'pizza': 5,
 'thor eat': 8,
 'eat pizza': 1,
 'loki': 2,
 'tail': 6,
 'loki tail': 4,
 'loki eat': 3}

In [14]:
v.transform(["Thor eat pizza"]).toarray()

array([[1, 1, 0, 0, 0, 1, 0, 1, 1]])

In [16]:
v.transform(["Hulk eat pizza"]).toarray()


array([[1, 1, 0, 0, 0, 1, 0, 0, 0]])

In [17]:
from google.colab import files
uploaded = files.upload()

Saving News_Category_Dataset_v3.json to News_Category_Dataset_v3.json


In [20]:
import pandas as pd
df=pd.read_json("News_Category_Dataset_v3.json", lines=True)
print(df.shape)
df.head()

(209527, 6)


,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [21]:
df.category.value_counts()


,count
category,
POLITICS,35602
WELLNESS,17945
ENTERTAINMENT,17362
TRAVEL,9900
STYLE & BEAUTY,9814
PARENTING,8791
HEALTHY LIVING,6694
QUEER VOICES,6347
FOOD & DRINK,6340


In [26]:
min_samples=1381
df_business=df[df.category=="BUSINESS"].sample(min_samples, random_state=2022)
df_sports=df[df.category=="SPORTS"].sample(min_samples, random_state=2022)
df_crime=df[df.category=="CRIME"].sample(min_samples, random_state=2022)
df_science=df[df.category=="SCIENCE"].sample(min_samples, random_state=2022)


In [27]:
df_balanced=pd.concat([df_business, df_sports, df_crime, df_science], axis=0)
df_balanced.category.value_counts()

,count
category,
BUSINESS,1381
SPORTS,1381
CRIME,1381
SCIENCE,1381


In [28]:
target={'BUSINESS':0,'SPORTS':1, 'CRIME':2, 'SCIENCE':3}
df_balanced['category_num']=df_balanced.category.map(target)

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_balanced.headline,
    df_balanced.category_num,
    test_size=0.2, # 20% samples will go to test dataset
    random_state=2022,
    stratify=df_balanced.category_num
)

In [32]:
print(df_balanced.columns)


Index(['link', 'headline', 'category', 'short_description', 'authors', 'date',
       'category_num'],
      dtype='object')


In [37]:
print(X_train.shape)
X_train.head()

(4419,)


,headline
196434,Space Plane: X-37B Video Shows Air Force Craft...
178136,Cave-Dwelling Plants: Strange Subterranean Net...
102479,Hockey Goalie Commits Humiliating Gaffe In Nat...
37965,Wisconsin Man Accused Of Sending Manifesto To ...
191964,"Derek McGlone, Teacher, Tries To Get Out Of Wo..."


In [38]:
y_train.value_counts()

,count
category_num,
3,1105
2,1105
0,1105
1,1104


In [42]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

clf=Pipeline([
    ('vectorizer_bow', CountVectorizer()),
    ('Multi NB', MultinomialNB())
])
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.76      0.83      0.79       276
           1       0.86      0.78      0.82       277
           2       0.84      0.89      0.87       276
           3       0.84      0.79      0.82       276

    accuracy                           0.82      1105
   macro avg       0.83      0.82      0.82      1105
weighted avg       0.83      0.82      0.82      1105



In [43]:
X_test[:5]

,headline
79832,Lender Discrimination May Be Pushing Black Chu...
106908,Mars One Candidate Thinks She Has What It Take...
166971,DNA: What Have You Done for Me Lately?
140653,Choose to Find Her and Mentor Her
111157,Death Of Mentally Ill Woman In Police Custody ...


In [44]:
y_test[:5]

,category_num
79832,0
106908,3
166971,3
140653,0
111157,2


In [45]:
y_pred[:5]

array([0, 3, 3, 2, 2])

In [48]:
df_balanced['preprocessed_txt']=df_balanced.headline.apply(preprocess)

In [49]:
df_balanced.head()

,link,headline,category,short_description,authors,date,category_num,preprocessed_txt
181516,https://www.huffingtonpost.com/entry/entrepren...,Entrepreneurism: Lots of Little Traumas and No...,BUSINESS,"I loved my years in corporate America, which I...","Liz Ryan, Contributor\nSpeaker, writer, sopran...",2012-11-25,0,entrepreneurism lot Little Traumas Big Ones
58552,https://www.huffingtonpost.com/entry/tesla-fas...,Tesla Just Unveiled The Quickest Car You Can A...,BUSINESS,A new battery upgrade extends the range of the...,"Alexandria Sage, Reuters",2016-08-23,0,Tesla unveil Quickest Car actually buy
155102,https://www.huffingtonpost.com/entry/workers-p...,90 Percent Of Employers Tie Workers' Pay To Co...,BUSINESS,Caterpillar will not break any profit records ...,"Reuters, Reuters",2013-09-01,0,90 percent Employers tie Workers pay company p...
71725,https://www.huffingtonpost.com/entry/us-tax-ha...,One Of Ben Carson's Craziest Ideas Is Coming True,BUSINESS,The U.S. is the world's hottest new tax haven.,Ben Walsh,2016-03-26,0,Ben Carson Craziest Ideas come true
80455,https://www.huffingtonpost.comhttp://www.bloom...,CEO Who Price Gouged HIV Drug Arrested For Sec...,BUSINESS,"A boyish drug company entrepreneur, who rocket...",,2015-12-17,0,ceo price gouge HIV Drug arrest Securities Fra...


In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_balanced.preprocessed_txt,
    df_balanced.category_num,
    test_size=0.2, # 20% samples will go to test dataset
    random_state=2022,
    stratify=df_balanced.category_num
)

In [51]:
clf=Pipeline([
    ('vectorizer_bow', CountVectorizer()),
    ('Multi NB', MultinomialNB())
])
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.78      0.75      0.76       276
           1       0.84      0.80      0.82       277
           2       0.80      0.92      0.86       276
           3       0.85      0.80      0.82       276

    accuracy                           0.82      1105
   macro avg       0.82      0.82      0.82      1105
weighted avg       0.82      0.82      0.82      1105

